In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import hashlib
import ast
import re

In [2]:
FILE = '2026-04-19.xlsx' 
path = Path()
file_path = path.absolute() / 'out' / FILE

In [3]:
EXCLUDED_WEIGHTS = [
    '[1, 0, 0, 0, 0]',
    '[0, 1, 0, 0, 0]',
    '[0, 0, 1, 0, 0]',
    '[0, 0, 0, 1, 0]',
    '[0, 0, 0, 0, 1]',
]

In [4]:
df = pd.read_excel(file_path, engine='openpyxl')

In [5]:
df = df[~df["weight"].isin(EXCLUDED_WEIGHTS)].copy()

In [6]:
df = df[~df['instancia'].isna()]

In [7]:
hash_list = df["weight_hash"].unique()

def make_weight_hash_map_from_list(hash_list, start_at=1):
    return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

hash_map = make_weight_hash_map_from_list(hash_list)

In [8]:
df["weight_label"] = df["weight_hash"].map(hash_map).fillna(df["weight_hash"])

In [9]:
df['instancia'] = df['instancia'].astype(int)

In [10]:
df['desv_rel_targ_1'] = df['p1'] / df['f1_target']
df['desv_rel_targ_2'] = df['p2'] / df['f2_target']
df['desv_rel_targ_3'] = df['p3'] / df['f3_target']
df['desv_rel_targ_4'] = df['p4'] / df['f4_target']
df['desv_rel_targ_5'] = df['p5'] / df['f5_target']

In [17]:
omegas = np.arange(0, 4, 0.001)
fob_cols = [f'desv_rel_targ_{i}' for i in range(1, 6)]
total_experiments = df.shape[0]

df_long = df.melt(
    # id_vars=['weight_label'],
    value_vars=fob_cols,
    var_name='fob',
    value_name='rtd_value'
)

out = (
    df_long.groupby(['fob'])['rtd_value']
    .apply(lambda s: pd.Series({omega: (s <= omega).sum() for omega in omegas}))
    .rename('count')
    .reset_index()
    .rename(columns={'level_1': 'omega'})
)

# out: weight | alpha | tau | count

In [18]:
out['count_norm'] = out['count'] / out['count'].max()

In [19]:
out

,fob,omega,count,count_norm
0,desv_rel_targ_1,0.000,10380,0.376578
1,desv_rel_targ_1,0.001,12963,0.470287
2,desv_rel_targ_1,0.002,13283,0.481897
3,desv_rel_targ_1,0.003,13377,0.485307
4,desv_rel_targ_1,0.004,13437,0.487484
...,...,...,...,...
19995,desv_rel_targ_5,3.995,26857,0.974351
19996,desv_rel_targ_5,3.996,26857,0.974351
19997,desv_rel_targ_5,3.997,26857,0.974351
19998,desv_rel_targ_5,3.998,26858,0.974387


In [22]:
out_plot = out.copy()

# Mapear desv_rel_targ_{i} para f<sup>i</sup> (HTML superscript)
out_plot['fob_label'] = out_plot['fob'].str.extract(r'(\d+)')[0].apply(lambda x: f'f<sup>{x}</sup>')

fig = px.line(out_plot[['omega', 'count_norm', 'fob_label']], x='omega', y='count_norm', color='fob_label', height=600, markers=False)

# Remover o título da legenda
fig.update_layout(
    coloraxis_colorbar=dict(title=""),
    legend=dict(title="")
)

# Configurar eixos
fig.update_xaxes(
    range=[0, None],
    title="RTD limit",
    showgrid=True,
    gridcolor='lightgray',
    showline=True,
    linewidth=2,
    linecolor='black'
)

fig.update_yaxes(
    title="RTD proportion",
    showgrid=True,
    gridcolor='lightgray',
    showline=True,
    linewidth=2,
    linecolor='black'
)

# Deixar fundo branco
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()
